In [ ]:
!pip install torch pandas scikit-learn onnx

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving processed.cleveland.data to processed.cleveland.data


In [ ]:
import pandas as pd

columns = [
    "age","sex","cp","trestbps","chol","fbs",
    "restecg","thalach","exang","oldpeak",
    "slope","ca","thal","target"
]

df = pd.read_csv("processed.cleveland.data", names=columns)


df.replace("?", pd.NA, inplace=True)
df.dropna(inplace=True)

df = df.apply(pd.to_numeric)


df["target"] = df["target"].apply(lambda x: 1 if x > 0 else 0)

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
import torch
import torch.nn as nn

class HeartModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(13, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = HeartModel()

In [ ]:
import torch.optim as optim

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 0.6956077814102173
Epoch 10, Loss: 0.661177933216095
Epoch 20, Loss: 0.604398787021637
Epoch 30, Loss: 0.5251574516296387
Epoch 40, Loss: 0.4480585753917694
Epoch 50, Loss: 0.3900121748447418
Epoch 60, Loss: 0.3492736220359802
Epoch 70, Loss: 0.31906452775001526
Epoch 80, Loss: 0.29518449306488037
Epoch 90, Loss: 0.27317342162132263


In [ ]:
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

with torch.no_grad():
    preds = model(X_test_tensor)
    preds = (preds > 0.5).float()
    accuracy = (preds == y_test_tensor).float().mean()

print("Accuracy:", accuracy.item())

Accuracy: 0.8833333253860474


In [ ]:
dummy_input = torch.randn(1, 13)

torch.onnx.export(
    model,
    dummy_input,
    "heart_model.onnx",
    input_names=["input"],
    output_names=["output"],
    export_params=True,
    do_constant_folding=True,
    opset_version=11
)

/tmp/ipykernel_7801/650675801.py:3: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
W0417 05:19:13.696000 7801 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `HeartModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `HeartModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.10.0+cpu',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,13]>
            ),
            outputs=(
                %"output"<FLOAT,[1,1]>
            ),
            initializers=(
                %"net.0.weight"<FLOAT,[64,13]>{TorchTensor(...)},
                %"net.0.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"net.2.bias"<FLOAT,[32]>{TorchTensor(...)},
                %"net.4.weight"<FLOAT,[16,32]>{TorchTensor(...)},
                %"net.4.bias"<FLOAT,[16]>{TorchTensor(...)},
                %"net.6.weight"<FLOAT,[1,16]>{TorchTensor(...)},
                %"net.6.bias"<FLOAT,[1]>{TorchTensor<FLOAT,[1]>(Parameter containing: tensor([0.2674], requires_grad=True), name='net.6.bias')},
         

In [ ]:
import onnx

model_onnx = onnx.load("heart_model.onnx")
onnx.save_model(model_onnx, "heart_model.onnx")